<a href="https://colab.research.google.com/github/Jamesstammers/SecurityTools/blob/main/CaseTemplate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 📥 Kibana Alert Parser
# @markdown Paste the raw JSON or redacted text from Kibana into the box below and run the cell.
Paste_Data_Here = "" # @param {type:"string"}
raw_text = Paste_Data_Here

import re
from IPython.display import HTML, display

if not raw_text.strip():
    print("❌ Error: Please paste your Kibana data into the input box on the right.")
else:
    # 1. FIELD EXTRACTION LIST
    fields_to_extract = [
        "kibana.alert.rule.name",
        "kibana.alert.rule.threat.tactic.name",
        "signal.rule.threat.technique.name",
        "kibana.alert.rule.threat.technique.id",
        "kibana.alert.rule.threat.technique.reference",
        "process.command_line",
        "process.parent.executable",
        "user.name.text",
        "host.name",
        "winlog.event_id",
        "kibana.alert.original_time",
        "kibana.alert.reason",
        "kibana.alert.rule.false_positives",
        "signal.rule.false_positives",
        "url.original",
        "source.enrichment.site_name_and_system",
        "destination.ip",
        "source.ip",
        "source.port",
        "destination.port",
        "destination.bytes",
        "user_agent.original",
        "event.action",
        "http.proxy.status_code",
        "hashicorp_vault.audit.request.headers.user-agent"
    ]

    results = {}
    for field in fields_to_extract:
        pattern = rf'"{re.escape(field)}":\s*(?:\[\s*)?"(.*?)"(?=\s*\]|\s*,)'
        match = re.search(pattern, raw_text, re.DOTALL)
        if match:
            val = match.group(1).replace('\\\\', '\\').replace('\\"', '"')
            results[field] = val
        else:
            results[field] = ""

    def is_valid(val):
        return val and val.strip() not in ["", "Not Found", "N/A"]

    # 2. ASSEMBLE THE MARKDOWN
    rule_title = results.get('kibana.alert.rule.name', 'Security Alert Investigation')
    md_lines = [f"# 🛡️ {rule_title}"]
    if is_valid(results.get('kibana.alert.reason')):
        md_lines.append(f"`{results['kibana.alert.reason']}`")
    md_lines += ["", "## 📋 Key Information", "| Field | Value |", "| :--- | :--- |"]

    if is_valid(results.get('host.name')): md_lines.append(f"| **Host Name** | `{results['host.name']}` |")
    if is_valid(results.get('user.name.text')): md_lines.append(f"| **User Name** | `{results['user.name.text']}` |")
    if is_valid(results.get('event.action')): md_lines.append(f"| **Action** | `{results['event.action']}` |")

    if is_valid(results.get('source.ip')):
        src = f"`{results['source.ip']}`"
        if is_valid(results.get('source.port')): src += f":`{results['source.port']}`"
        md_lines.append(f"| **Source** | {src} |")
    if is_valid(results.get('destination.ip')):
        dst = f"`{results['destination.ip']}`"
        if is_valid(results.get('destination.port')): dst += f":`{results['destination.port']}`"
        md_lines.append(f"| **Destination** | {dst} |")
    if is_valid(results.get('destination.bytes')): md_lines.append(f"| **Bytes Sent** | `{results['destination.bytes']}` |")
    if is_valid(results.get('url.original')): md_lines.append(f"| **URL** | `{results['url.original']}` |")
    if is_valid(results.get('http.proxy.status_code')): md_lines.append(f"| **Proxy Status** | `{results['http.proxy.status_code']}` |")
    if is_valid(results.get('user_agent.original')): md_lines.append(f"| **User Agent** | `{results['user_agent.original']}` |")
    if is_valid(results.get('hashicorp_vault.audit.request.headers.user-agent')): md_lines.append(f"| **Vault UA** | `{results['hashicorp_vault.audit.request.headers.user-agent']}` |")
    if is_valid(results.get('source.enrichment.site_name_and_system')): md_lines.append(f"| **Site/System** | `{results['source.enrichment.site_name_and_system']}` |")

    if is_valid(results.get('signal.rule.threat.technique.name')):
        tech_id = results.get('kibana.alert.rule.threat.technique.id', '')
        md_lines.append(f"| **MITRE Technique** | {results['signal.rule.threat.technique.name']} ({tech_id}) |")
    if is_valid(results.get('kibana.alert.rule.threat.tactic.name')): md_lines.append(f"| **MITRE Tactic** | {results['kibana.alert.rule.threat.tactic.name']} |")
    if is_valid(results.get('kibana.alert.rule.threat.technique.reference')): md_lines.append(f"| **MITRE Link** | [View on MITRE ATT&CK]({results['kibana.alert.rule.threat.technique.reference']}) |")
    if is_valid(results.get('winlog.event_id')): md_lines.append(f"| **Event ID** | `{results['winlog.event_id']}` |")
    if is_valid(results.get('kibana.alert.original_time')): md_lines.append(f"| **Alert Time** | `{results['kibana.alert.original_time']}` |")

    if is_valid(results.get('process.parent.executable')):
        md_lines += ["", "**Parent Process:**", "```powershell", f"{results['process.parent.executable']}", "```"]
    if is_valid(results.get('process.command_line')):
        md_lines += ["", "**Command Line:**", "```powershell", f"{results['process.command_line']}", "```"]

    triage_step_3 = "3. **Check against False Positives:** Verify if activity matches baseline activity."
    fps = results.get('kibana.alert.rule.false_positives') or results.get('signal.rule.false_positives')
    if is_valid(fps): triage_step_3 += f" Known false positives for this rule include: {fps}"

    md_lines += [
        "", "## ⚠️ Activity Type Detected", "- [ ] Malware", "- [ ] Hacking", "- [ ] Social", "- [ ] Misuse", "- [ ] Physical", "- [ ] Error",
        "", "## 📅 Timeline of Events", "| Timestamp | Event Description |", "| :--- | :--- |", "| `[HH:MM:SS]` | *Prior Activity...* |",
        f"| `{results.get('kibana.alert.original_time', 'T0')}` | **ALERT TRIGGERED** |", "| `[HH:MM:SS]` | *Follow-on Activity...* |",
        "", "## 🎯 Potential Impact", "Evaluate the potential damage or risk to operations, data, and reputation:", "- **Operations:** (e.g., Service downtime, system lockdown)", "- **Data:** (e.g., Potential for exfiltration, unauthorized modification)", "- **Reputation:** (e.g., Impact on customer trust, regulatory compliance)",
        "", "## 🔍 Triage and Analysis Steps", "1. **Consult Guide:** Refer to the official Investigation Guide.", "2. **Verify Context:** Determine if the activity matches known administrative patterns or typical user behaviour.", triage_step_3,
        "", "## 🏁 Summary, Conclusion, and Next Steps", "**Final Determination:** (Benign / True Positive / False Positive)", "", "**Summary:**", "", "", "", "**Next Steps:**", "", "- [ ] Incident escalation required", "- [ ] Suppress alert / Tune rule", "- [ ] Close case"
    ]

    final_markdown = "\n".join(md_lines)

    # 3. RENDER OUTPUT BOX WITH FEEDBACK BUTTON (NO BROWSER ALERT)
    html_template = f"""
    <div style="background-color: #f8f9fa; border: 1px solid #dee2e6; border-radius: 8px; padding: 15px; font-family: sans-serif;">
        <button id="copyBtn" onclick="copyToClipboard()" style="background-color: #007bff; color: white; border: none; padding: 8px 15px; border-radius: 4px; cursor: pointer; margin-bottom: 10px; font-weight: bold; transition: 0.3s;">📋 Copy Case Template</button>
        <textarea id="markdownOutput" style="width: 100%; height: 300px; border: 1px solid #ced4da; border-radius: 4px; padding: 10px; font-family: monospace; font-size: 12px; background-color: white;">{final_markdown}</textarea>
    </div>
    <script>
    function copyToClipboard() {{
        var copyText = document.getElementById("markdownOutput");
        var btn = document.getElementById("copyBtn");
        copyText.select();
        copyText.setSelectionRange(0, 99999);
        document.execCommand("copy");

        // Visual feedback without alert
        btn.innerHTML = "✅ Copied!";
        btn.style.backgroundColor = "#28a745";

        setTimeout(function() {{
            btn.innerHTML = "📋 Copy Case Template";
            btn.style.backgroundColor = "#007bff";
        }}, 2000);
    }}
    </script>
    """
    display(HTML(html_template))
